In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import joblib, json, warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('student_performance.csv')
print(f"Total rows: {len(df):,}")
df.head()

Total rows: 1,000,000


,student_id,weekly_self_study_hours,attendance_percentage,class_participation,total_score,grade
0,1,18.5,95.6,3.8,97.9,A
1,2,14.0,80.0,2.5,83.9,B
2,3,19.5,86.3,5.3,100.0,A
3,4,25.7,70.2,7.0,100.0,A
4,5,13.4,81.9,6.9,92.0,A


In [3]:
df.info(
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 6 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   student_id               1000000 non-null  int64  
 1   weekly_self_study_hours  1000000 non-null  float64
 2   attendance_percentage    1000000 non-null  float64
 3   class_participation      1000000 non-null  float64
 4   total_score              1000000 non-null  float64
 5   grade                    1000000 non-null  object 
dtypes: float64(4), int64(1), object(1)
memory usage: 45.8+ MB


# Feature Engneering

In [4]:

df['score_x_study'] = df['total_score'] * df['weekly_self_study_hours']
df['study_sq']      = df['weekly_self_study_hours'] ** 2
df['score_sq']      = df['total_score'] ** 2
df['score_bin']     = pd.cut(
    df['total_score'],
    bins=[0, 60, 70, 80, 90, 100],
    labels=[0, 1, 2, 3, 4]
).astype(float)

In [5]:
FEATURE_COLS = [
    'weekly_self_study_hours',
    'attendance_percentage',
    'class_participation',
    'total_score',
    'score_x_study',
    'study_sq',
    'score_sq',
    'score_bin',
]

X = df[FEATURE_COLS].values
y = df['grade'].values

In [6]:
#  ENCODE + SPLIT
le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Classes:", le.classes_)
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.20, random_state=42, stratify=y_enc
)

Classes: ['A' 'B' 'C' 'D' 'F']


In [7]:
# SCALE
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)


In [8]:
class_weights_arr = compute_class_weight(
    'balanced', classes=np.unique(y_enc), y=y_enc
)
class_weight_dict = {i: w for i, w in enumerate(class_weights_arr)}
print("Class weights:", {le.classes_[k]: round(v, 2) for k, v in class_weight_dict.items()})


Class weights: {'A': np.float64(0.36), 'B': np.float64(0.77), 'C': np.float64(1.41), 'D': np.float64(4.44), 'F': np.float64(32.24)}


# Build Model


In [9]:
model = keras.Sequential([
    layers.Input(shape=(len(FEATURE_COLS),)),
 
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
 
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
 
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.1),
 
    layers.Dense(64, activation='relu'),
 
    layers.Dense(5, activation='softmax'),   # 5 classes: A B C D F
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │         4,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 180,997 (707.02 KB)

 Trainable params: 179,205 (700.02 KB)

 Non-trainable params: 1,792 (7.00 KB)

In [10]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=4096,
    validation_split=0.10,
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=4,
            restore_best_weights=True,
            monitor='val_accuracy'
        ),
        keras.callbacks.ReduceLROnPlateau(
            patience=3,
            factor=0.5,
            monitor='val_loss'
        ),
    ],
    verbose=1,
)

Epoch 1/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 31s 151ms/step - accuracy: 0.9460 - loss: 0.2056 - val_accuracy: 0.7851 - val_loss: 0.6338 - learning_rate: 0.0010
Epoch 2/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 26s 146ms/step - accuracy: 0.9824 - loss: 0.0872 - val_accuracy: 0.9424 - val_loss: 0.2014 - learning_rate: 0.0010
Epoch 3/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 28s 157ms/step - accuracy: 0.9871 - loss: 0.0620 - val_accuracy: 0.9767 - val_loss: 0.0654 - learning_rate: 0.0010
Epoch 4/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 28s 156ms/step - accuracy: 0.9895 - loss: 0.0531 - val_accuracy: 0.9952 - val_loss: 0.0136 - learning_rate: 0.0010
Epoch 5/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 29s 164ms/step - accuracy: 0.9907 - loss: 0.0475 - val_accuracy: 0.9964 - val_loss: 0.0114 - learning_rate: 0.0010
Epoch 6/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 27s 153ms/step - accuracy: 0.9914 - loss: 0.0424 - val_accuracy: 0.9954 - val_loss: 0.0117 - learning_rate: 0.0010
Epoch 7/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 30s 171ms/step - accuracy: 0.9

In [11]:
y_pred = np.argmax(model.predict(X_test, batch_size=8192), axis=1)
acc    = accuracy_score(y_test, y_pred)

print(f"\n{'='*50}")
print(f"  ✅ Test Accuracy: {acc*100:.2f}%")
print(f"{'='*50}\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step

  ✅ Test Accuracy: 99.68%

              precision    recall  f1-score   support

           A       1.00      1.00      1.00    109729
           B       1.00      1.00      1.00     51635
           C       1.00      0.99      0.99     28396
           D       0.98      1.00      0.99      8999
           F       0.98      0.99      0.99      1241

    accuracy                           1.00    200000
   macro avg       0.99      1.00      0.99    200000
weighted avg       1.00      1.00      1.00    200000

